In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [5]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [11]:
X_train,X_val,y_train,y_val = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [18]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)

ss = StandardScaler()
X_train = ss.fit_transform(X_train)
X_val = ss.transform(X_val)

In [20]:
X_train_tensors = torch.from_numpy(X_train)
X_val_tensors = torch.from_numpy(X_val)
y_train_tensors = torch.from_numpy(y_train)
y_val_tensors = torch.from_numpy(y_val)

In [22]:
X_train_tensors.shape

torch.Size([455, 30])

In [66]:
y_train_tensors.shape

torch.Size([455])

# Model Building

In [114]:
class NN():

  def __init__(self, X):

    self.wt = torch.rand(X.shape[1], dtype=torch.float64, requires_grad=True)
    self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

  def forward_pass(self, X):
    z = torch.matmul(X, self.wt) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred)).mean()
    return loss





In [115]:
model = NN(X_train)
lr = 0.1
epochs = 25

for _ in range(epochs):
  y_pred = model.forward_pass(X_train_tensors)
  loss = model.loss_function(y_pred,y_train_tensors)
  loss.backward()

  with torch.no_grad():
    model.wt -= lr*model.wt.grad
    model.bias -= lr*model.bias.grad

  model.wt.grad.zero_()
  model.bias.grad.zero_()

  print(f"for epoch{_} the loss is {loss}")

for epoch0 the loss is 0.6825619219731789
for epoch1 the loss is 0.665810618644093
for epoch2 the loss is 0.6496247682009656
for epoch3 the loss is 0.6336098192136965
for epoch4 the loss is 0.6172911372815147
for epoch5 the loss is 0.601483377698503
for epoch6 the loss is 0.5861615142454846
for epoch7 the loss is 0.5713012472063355
for epoch8 the loss is 0.5568791123259615
for epoch9 the loss is 0.5428726224703466
for epoch10 the loss is 0.5292604103584428
for epoch11 the loss is 0.5160223483914379
for epoch12 the loss is 0.503139631623787
for epoch13 the loss is 0.49059482012590244
for epoch14 the loss is 0.4783718431931007
for epoch15 the loss is 0.46645597238984193
for epoch16 the loss is 0.4548337724995698
for epoch17 the loss is 0.4434930389011344
for epoch18 the loss is 0.4324227292579763
for epoch19 the loss is 0.421612895567771
for epoch20 the loss is 0.41105462154844696
for epoch21 the loss is 0.4007399683048131
for epoch22 the loss is 0.39066193040786623
for epoch23 the loss 

In [118]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward_pass(X_val_tensors)
  y_pred = (y_pred > 0.999).float()
  accuracy = (y_pred == y_val_tensors).float().mean()
  print(f'Accuracy: {accuracy.item()}')


Accuracy: 0.8508771657943726


In [87]:
y_pred

tensor([0., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 1.,
        1., 1., 0., 0., 0., 0., 0., 0., 1., 1., 0., 1., 0., 0., 0., 0., 0., 1.,
        1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
        0., 0., 1., 0., 0., 0., 0., 1., 1., 1., 0., 1., 0., 0., 0., 1., 1., 0.,
        0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 1., 0., 1., 0., 0., 0.,
        1., 0., 0., 0., 0., 0.])